# 🚀 AI Blog Generator on Google Colab
Generate professional blog posts using free GPU power!

**No local compute needed** — everything runs on Colab's free GPU/TPU

## Setup Phase (Run once per session)

In [ ]:
# Clone the repository
!git clone https://github.com/YOUR_USERNAME/ai-blog-generator.git
%cd ai-blog-generator
!pwd

In [ ]:
# Install Python dependencies
!pip install -q langgraph langchain langchain-ollama langchain-core pydantic streamlit python-dotenv requests duckduckgo-search

In [ ]:
# Install Ollama (LLM that runs on Colab GPU)
!curl -fsSL https://ollama.ai/install.sh | sh

In [ ]:
# Start Ollama server in background
import subprocess
import time
import requests

# Start Ollama
proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print("⏳ Starting Ollama server...")
time.sleep(5)

# Wait for server to be ready
for i in range(10):
    try:
        requests.get('http://localhost:11434/api/tags')
        print("✅ Ollama server is ready")
        break
    except:
        time.sleep(1)
        print(f"  Waiting... ({i+1}/10)")

In [ ]:
# Download Mistral model (7B, fast, good quality)
# This takes ~2-3 minutes on first run
print("📥 Downloading Mistral model (~4GB)...")
print("This is a one-time download.\n")

!ollama pull mistral

print("\n✅ Model ready! You can now generate blogs.")

## Test LLM Connection

In [ ]:
# Test that everything works
from langchain_ollama import OllamaLLM

llm = OllamaLLM(
    model="mistral",
    base_url="http://localhost:11434",
    temperature=0.7
)

# Quick test
result = llm.invoke("Say 'Blog generator is ready!' in 3 words")
print("LLM Response:", result)
print("\n✅ Everything is working!")

## Generate Your Blog

In [ ]:
# Import the blog generator
from backend.pipeline import create_pipeline
from backend.models import GraphState

print("✅ Pipeline imported successfully")

In [ ]:
# Configure your blog
BLOG_TOPIC = "The Future of Artificial Intelligence"  # ← CHANGE THIS
BLOG_MODE = "open_book"  # "open_book" (with research) or "closed_book" (LLM only)

print(f"📝 Generating blog about: '{BLOG_TOPIC}'")
print(f"📚 Mode: {BLOG_MODE}")
print("\nThis will take 2-5 minutes...\n")

# Create and run pipeline
pipeline = create_pipeline(mode=BLOG_MODE)

initial_state = GraphState(
    topic=BLOG_TOPIC,
    mode=BLOG_MODE,
    errors=[]
)

result = pipeline.invoke(initial_state)

print("\n✅ Blog generated!")

In [ ]:
# Display the generated blog
blog_content = result.get("blog", "No blog generated")

print("\n" + "="*80)
print("📖 GENERATED BLOG")
print("="*80 + "\n")
print(blog_content)

In [ ]:
# Save the blog to a markdown file
import os

filename = "generated_blog.md"

with open(filename, "w") as f:
    f.write(blog_content)

print(f"✅ Blog saved to {filename}")
print(f"📊 Size: {len(blog_content)} characters")

In [ ]:
# Download the blog to your computer
from google.colab import files

print("💾 Downloading blog to your computer...")
files.download('generated_blog.md')
print("✅ Done!")

## Advanced: Generate Multiple Blogs

In [ ]:
# Generate multiple blogs in one session
topics = [
    "Machine Learning in Healthcare",
    "Renewable Energy Solutions",
    "Web3 and Blockchain"
]

for i, topic in enumerate(topics, 1):
    print(f"\n{'='*60}")
    print(f"[{i}/{len(topics)}] Generating: {topic}")
    print('='*60)
    
    pipeline = create_pipeline(mode="open_book")
    result = pipeline.invoke(GraphState(topic=topic, mode="open_book", errors=[]))
    
    # Save with timestamp
    import datetime
    filename = f"blog_{i}_{topic.replace(' ', '_')}.md"
    
    with open(filename, "w") as f:
        f.write(result.get("blog", ""))
    
    print(f"✅ Saved: {filename}")

print(f"\n✅ All {len(topics)} blogs generated!")

## Troubleshooting

In [ ]:
# Check system status
import subprocess
import requests

print("🔍 System Status:")
print()

# Check Ollama
try:
    resp = requests.get('http://localhost:11434/api/tags')
    models = resp.json().get('models', [])
    print(f"✅ Ollama: Running")
    print(f"   Available models: {[m['name'] for m in models]}")
except:
    print("❌ Ollama: Not responding")

# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "GPU not available"